In [20]:
import numpy as np
import sys
sys.path.append("../src")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import glob
import os
from utils import time_embedding_np, reward_simulate,simulate_data_raw,make_weighted_target,position_encoder
from utils import best_next_state,path_opt,preward_opt,compute_normalized_future_rewards
from treeple.experimental import StreamDecisionForest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import xgboost as xgb

In [21]:
base_reward = 10.0
map_name = "short"
decay_rate = 0.6
reward_period = 10 
session_duration = 1000000
rewards_in_period = []

total_steps = session_duration
for period_start in range(0, total_steps, reward_period):
    period_end = min(period_start + reward_period, total_steps)
    steps_in_period = np.arange(period_start, period_end)
    rewards_in_period.extend(base_reward * (decay_rate ** (steps_in_period - period_start)))

pattern = [1,1,1,1,1,1,1,2,3,4,5,5,5,5,5,5,5,4,3,2]
optimal_states = np.array(pattern*(session_duration // len(pattern))).reshape(-1,1)

In [22]:
def get_candidates(state, t):
    if state == 0:
        return [1, 0]
    elif state == 6:
        return [5, 6]
    else:
        # default 3-way split for illustration
        return [state-1, state, state+1]

def enumerate_paths(x0, t_inital = 0, t_prime=20):
    paths = [[x0]]
    for t in range(t_prime):
        new_paths = []
        for path in paths:
            curr = path[-1]
            for nxt in get_candidates(curr, t):
                new_paths.append(path + [nxt])
        paths = new_paths
    
    # Convert to DataFrame: each row is one path, columns t=0..T
    cols = [f"t={t_inital+i}" for i in range(t_prime+1)]
    df = pd.DataFrame(paths, columns=cols)
    return df


In [26]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden=(256, 128), p_dropout=0.2, use_sigmoid=True):
        super().__init__()
        layers = []
        d = input_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(p_dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        if use_sigmoid:           # keep preds in [0,1] if y is normalized
            layers += [nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)        # (N, 1)

def mse_loss(pred, target, eps=1e-12):
    # return torch.sqrt(nn.functional.mse_loss(pred, target) + eps)
    return nn.functional.mse_loss(pred, target)

def train_regressor(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_tr = float("inf")
    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        tot_rmse, n_batches = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float().view(-1, 1)  # ensure (N,1)

            opt.zero_grad()
            preds = model(xb)
            loss = mse_loss(preds, yb)
            loss.backward()
            opt.step()

            tot_rmse += loss.item()
            n_batches += 1

        # validation
        model.eval()
        val_rmse, m = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device).float()
                yb = yb.to(device).float().view(-1, 1)
                preds = model(xb)
                val_rmse += mse_loss(preds, yb).item()
                m += 1
        val_rmse /= max(1, m)
        tr_rmse = tot_rmse / max(1, n_batches)
        if ep % 500 == 0:
            print(f"Epoch {ep:02d} | train MSE {tr_rmse:.6f} | val MSE {val_rmse:.6f}")

        if tr_rmse <= best_tr:
            if val_rmse <= best_val: ## Ignore the posibility of overfitting
                best_tr = tr_rmse
                best_val = val_rmse
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                print(f"Best Model: Epoch {ep:02d} | train MSE {tr_rmse:.6f} | val MSE {val_rmse:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict(model, X):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        if isinstance(X, torch.Tensor):
            X_t = X.to(device).float()
        else:
            X_t = torch.from_numpy(np.asarray(X, dtype=np.float32)).to(device)
        preds = model(X_t).cpu().squeeze(1)
    return preds

In [ ]:
def train_model(X, Y, model= None, model_name = 'rf'):
    """Train or continue training the XGBRegressor."""
    if model is None:
        if model_name == 'rf':
            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                tree_method="exact",
                n_jobs=-1,
                learning_rate=0.05, max_depth=10,
                subsample=1, colsample_bytree=0.8,
                n_estimators=1000, eval_metric="rmse"
            )
            model.fit(X, Y)
        elif model_name == 'nn':
            n = X.shape[0]
            tr_size = int(X.shape[1]*0.9)
            val_size = X.shape[1] - tr_size

            idx_val = np.random.choice(n, val_size, replace=False)
            idx_tr  = np.setdiff1d(np.arange(n), idx_val)
            
            X_train = X[idx_tr,:]
            Y_train = np.array(Y[idx_tr]).reshape(-1,1)
            X_val = X[idx_val,:]
            Y_val = np.array(Y[idx_val]).reshape(-1,1)
            batch_size_train = 32
            batch_size_val = val_size

            train_ds = TensorDataset(torch.from_numpy(X_train.astype(np.float32)), torch.from_numpy(Y_train.astype(np.float32)))
            val_ds   = TensorDataset(torch.from_numpy(X_val.astype(np.float32)), torch.from_numpy(Y_val.astype(np.float32)))
            train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, drop_last=False)
            val_loader   = DataLoader(val_ds,  batch_size=batch_size_val, shuffle=False, drop_last=False)
            model = MLPRegressor(input_dim=X_train.shape[1], hidden=(128,128), p_dropout=0.1, use_sigmoid=False)
            model = train_regressor(model, train_loader, val_loader, epochs=1500, lr=1e-2,weight_decay = 1e-4)
    else:
        if model_name == 'rf':
            model.fit(X, Y)  # incremental update

        elif model_name == 'nn':
            tr_size = int(X.shape[1]*0.9)
            val_size = X.shape[1] - tr_size
            X_train = X[:tr_size,:]
            Y_train = np.array(Y[:tr_size]).reshape(-1,1)
            X_val = X[tr_size:tr_size+val_size,:]
            Y_val = np.array(Y[tr_size:tr_size+val_size]).reshape(-1,1)
            batch_size_train = 32
            batch_size_val = val_size

            train_ds = TensorDataset(torch.from_numpy(X_train.astype(np.float32)), torch.from_numpy(X_train.astype(np.float32)))
            val_ds   = TensorDataset(torch.from_numpy(X_val.astype(np.float32)), torch.from_numpy(X_val.astype(np.float32)))
            train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, drop_last=False)
            val_loader   = DataLoader(val_ds,  batch_size=batch_size_val, shuffle=False, drop_last=False)
            model = MLPRegressor(input_dim=X_train.shape[1], hidden=(256,128), p_dropout=0.2, use_sigmoid=True)
            model = train_regressor(model, train_loader, val_loader, epochs=300, lr=3e-4,weight_decay = 1e-5)
    return model

def predict_paths(model, df_paths, t_start, t_prime,model_name = 'nn'):
    n_paths = df_paths.shape[0]
    preds = np.zeros((n_paths, t_prime+1))
    for i in range(t_prime+1):
        cand_pe = position_encoder(df_paths.iloc[:,i], type="onehot")
        curr_time = t_start + 1 + i
        time_emb = time_embedding_np(np.ones(cand_pe.shape[0]) * curr_time, tdim=50)
        features = np.hstack([cand_pe, time_emb])
        if model_name == 'nn':
            preds[:, i] = predict(model, features)
        else:
            preds[:, i] = model.predict(features)
    return preds

def choose_next_state(model, current_state, t_now, t_prime, gamma,model_name = 'nn'):
    """Pick best next state based on discounted rewards."""
    df_paths = enumerate_paths(x0=current_state, t_inital=t_now, t_prime=t_prime)
    preds = predict_paths(model, df_paths, t_now, t_prime,model_name = model_name)

    rewards_matrix = preds[:, 1:]   # ignore first col (current)
    discounts = gamma ** np.arange(1, t_prime+1)
    total_disc = (rewards_matrix * discounts).sum(axis=1)

    best_idx = np.argmax(total_disc)
    return int(df_paths.iloc[best_idx, 1]), df_paths, preds

def run_inference(model, start_state, T_p, t_prime, gamma, rewards_in_period,model_name):
    """Generate future trajectory and compute regret."""
    current_state = start_state
    future_states = [current_state]

    for delta_t in range(100):
        next_state, df_paths, preds = choose_next_state(model, current_state, T_p + delta_t, t_prime, gamma,model_name)
        future_states.append(next_state)
        current_state = next_state

    # compute rewards
    irewards_test = [reward_simulate(future_states[i], T_p+i, rewards_in_period) 
                     for i in range(1, len(future_states))]
    preward = compute_normalized_future_rewards(irewards_test, 100, gamma)

    # optimal path benchmark
    path, ireward_opt = path_opt(future_states[0], T_p, 100)
    preward_opt_test = compute_normalized_future_rewards(ireward_opt[1:], 100, gamma)

    pregret = (np.sum(preward_opt_test).item() - np.sum(preward).item()) / 100

    return pregret


T = 500
gamma = 0.5
t_prime = 6
PREGRETS = []

for rep in range(1):
    # one simulation per rep
    actions, states, irewards, times = simulate_data_raw(
        rewards_in_period=rewards_in_period,
        session_duration=2000,
        tdim=50, n_sessions=1, seed=515+rep
    )

    # initial dataset
    X_train = np.hstack([position_encoder(states[:T,0], type="onehot"), states[:T,1:]])
    Y_train = irewards[:T]
    model = train_model(X_train, Y_train,model_name = 'nn')

    PREGRET = []
    # --- online training ---
    current_state = states[T,0]
    for delta_t in range(200):
        next_state, df_paths, preds = choose_next_state(model, current_state, T+delta_t, t_prime, gamma,'nn')
        next_time = T+1+delta_t
        next_features = np.hstack([
            position_encoder(np.array([next_state]), type="onehot"),
            time_embedding_np(next_time, tdim=50).reshape(1,-1)
        ])
        next_reward = reward_simulate(next_state, next_time-1, rewards_in_period)

        X_train = np.vstack([X_train, next_features])
        Y_train = np.vstack([Y_train.reshape(-1,1), [[next_reward]]])
        # X_train = np.hstack([position_encoder(states[:T+delta_t,0], type="onehot"), states[:T+delta_t,1:]])
        # Y_train = irewards[:T+delta_t]
        model = train_model(X_train, Y_train, model_name = 'nn')
        current_state = next_state  # move forward

    
        T_p = delta_t + T
        start_state = current_state
        pregret_val = run_inference(model, start_state, T_p, t_prime, gamma, rewards_in_period,'nn')
        print(f"rep {rep}, T_train {T_p}, pregret={pregret_val:.4f}")
        PREGRET.append(pregret_val)

    PREGRETS.append(PREGRET)


Best Model: Epoch 01 | train MSE 2.742399 | val MSE 0.191310
Best Model: Epoch 04 | train MSE 1.168337 | val MSE 0.036710
Best Model: Epoch 07 | train MSE 0.566451 | val MSE 0.008521
Epoch 500 | train MSE 0.071049 | val MSE 0.070503
Best Model: Epoch 541 | train MSE 0.063530 | val MSE 0.006153
Epoch 1000 | train MSE 0.123844 | val MSE 0.019158
Best Model: Epoch 1259 | train MSE 0.049872 | val MSE 0.005621
Best Model: Epoch 1261 | train MSE 0.048488 | val MSE 0.004960
Best Model: Epoch 1283 | train MSE 0.034601 | val MSE 0.004303
Best Model: Epoch 1351 | train MSE 0.030720 | val MSE 0.004167
Epoch 1500 | train MSE 0.164171 | val MSE 0.004136
Best Model: Epoch 01 | train MSE 2.635610 | val MSE 0.103639
Best Model: Epoch 02 | train MSE 1.678970 | val MSE 0.010503
Best Model: Epoch 03 | train MSE 0.934265 | val MSE 0.000573
Best Model: Epoch 05 | train MSE 0.612826 | val MSE 0.000050
Best Model: Epoch 07 | train MSE 0.520656 | val MSE 0.000022


/var/folders/7_/9pm8yyb11yl_smkwv7tk0ysc0000gn/T/ipykernel_26614/2581025797.py:66: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  preds[:, i] = predict(model, features)*10


Best Model: Epoch 132 | train MSE 0.112945 | val MSE 0.000010
Best Model: Epoch 270 | train MSE 0.082139 | val MSE 0.000008
Best Model: Epoch 325 | train MSE 0.074402 | val MSE 0.000003
Epoch 500 | train MSE 0.031021 | val MSE 0.000000
Best Model: Epoch 500 | train MSE 0.031021 | val MSE 0.000000
Epoch 1000 | train MSE 0.057497 | val MSE 0.000700
Epoch 1500 | train MSE 0.078408 | val MSE 0.000063
rep 0, T_train 500, pregret=1.0777
Best Model: Epoch 01 | train MSE 2.557951 | val MSE 15.283788
Best Model: Epoch 02 | train MSE 1.735827 | val MSE 13.298276
Best Model: Epoch 03 | train MSE 0.932600 | val MSE 9.320078
Best Model: Epoch 11 | train MSE 0.158294 | val MSE 8.712472
Best Model: Epoch 21 | train MSE 0.155539 | val MSE 7.914013
Best Model: Epoch 79 | train MSE 0.128355 | val MSE 7.057790
Best Model: Epoch 88 | train MSE 0.110310 | val MSE 6.330640
Epoch 500 | train MSE 0.042757 | val MSE 10.647520
Best Model: Epoch 529 | train MSE 0.082022 | val MSE 6.121145
Best Model: Epoch 534 |

KeyboardInterrupt: 